In [0]:
import uuid
from datetime import datetime
from pyspark.sql import SparkSession

class PipelineLogger:
    def __init__(self, spark: SparkSession, pipeline_name: str,run_id: int ):
        self.spark = spark
        self.pipeline_name = pipeline_name
        self.run_id = run_id
        self.start_time = datetime.now()
        
        
    def open_runlog(self):
        """Creates an initial RUNNING entry in audit_runlog."""
        self.spark.sql(f"""
            INSERT INTO shprod.audit_runlog
            VALUES (
                '{self.run_id}',
                current_date(), 
                '{self.pipeline_name}', 
                'RUNNING', 
                TIMESTAMP('{self.start_time.isoformat()}'), 
                NULL, 
                NULL, 
                current_user()
            )
        """)

    def close_runlog(self, status: str = "COMPLETED"):
        """Updates runlog on pipeline completion or failure."""
        end_time = datetime.now()
        duration = int((end_time - self.start_time).total_seconds())
        self.spark.sql(f"""
            UPDATE shprod.audit_runlog
            SET status = '{status}',
                end_time = TIMESTAMP('{end_time.isoformat()}'),
                total_duration_sec = {duration}
            WHERE run_id = '{self.run_id}'
        """)

    def update_active_dt(self):
        """Updates the active date for a table in audit_active_dates."""
        self.spark.sql(f"""
            UPDATE shprod.stg_payroll_employee_nyc
            SET active_dt = current_date()
            WHERE run_id = '{self.run_id}'
        """)

    def log_step(self, step_name: str, source_table: str, target_table: str,error_table : str, 
                 status: str, rows_read: int = 0, rows_inserted: int = 0, 
                 rows_updated: int = 0, rows_quarantined: int = 0, 
                 start_time: datetime = None, end_time: datetime = None, 
                 error_msg: str = None):
        """Inserts a record into audit_execution_logs."""
        log_id = str(uuid.uuid4())
        err_str = f"'{error_msg}'" if error_msg else "NULL"
        start_ts = f"TIMESTAMP('{start_time.isoformat()}')" if start_time else "current_timestamp()"
        end_ts = f"TIMESTAMP('{end_time.isoformat()}')" if end_time else "current_timestamp()"
        
        self.spark.sql(f"""
            INSERT INTO shprod.audit_execution_logs
            VALUES (
                '{log_id}',
                '{self.run_id}',
                current_date(),
                '{step_name}',
                '{source_table}',
                '{target_table}',
                '{error_table}',
                '{status}',
                {rows_read},
                {rows_inserted},
                {rows_updated},
                {rows_quarantined},
                {start_ts},
                {end_ts},
                {err_str}
            )
        """)